In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Libraries imported Successfully!")

Libraries imported Successfully!


In [2]:
customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")

In [3]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Category Translation:", category_translation.shape)

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Category Translation: (71, 2)


In [5]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)
orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"]
)
orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"]
)
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)
orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"]
)
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)
reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

In [6]:
reviews["review_comment_title"] = reviews[
    "review_comment_title"
].fillna("No Comment")

reviews["review_comment_message"] = reviews[
    "review_comment_message"
].fillna("No Comment")

products["product_category_name"] = products[
    "product_category_name"
].fillna("Unknown")

In [7]:
order_sales = (
    order_items
    .groupby("order_id")
    .agg(
        total_sales=("price", "sum"),
        total_freight=("freight_value", "sum"),
        total_items=("order_item_id", "count")
    )
    .reset_index()
)

In [8]:
order_analysis = orders.merge(
    order_sales,
    on="order_id",
    how="left"
)

In [9]:
order_analysis = order_analysis.merge(
    customers[["customer_id", "customer_state", "customer_city"]],
    on="customer_id",
    how="left"
)

In [10]:
print(order_analysis.shape)
order_analysis.head()

(99441, 13)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_sales,total_freight,total_items,customer_state,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,1.00,SP,sao paulo
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,1.00,BA,barreiras
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,1.00,GO,vianopolis
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,1.00,RN,sao goncalo do amarante
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,1.00,SP,santo andre


In [11]:
print("Descriptive Statistics - Order Level Data")

order_analysis[
    ["total_sales", "total_freight", "total_items"]
].describe()

Descriptive Statistics - Order Level Data


,total_sales,total_freight,total_items
count,"98,666.00","98,666.00","98,666.00"
mean,137.75,22.82,1.14
std,210.65,21.65,0.54
min,0.85,0.00,1.00
25%,45.90,13.85,1.00
50%,86.90,17.17,1.00
75%,149.90,24.04,1.00
max,"13,440.00","1,794.96",21.00


In [12]:
print("Descriptive Statistics - Order Item Data")

order_items[
    ["price", "freight_value"]
].describe()

Descriptive Statistics - Order Item Data


,price,freight_value
count,"112,650.00","112,650.00"
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,"6,735.00",409.68


In [13]:
print("Descriptive Statistics - Payments")

payments[
    ["payment_installments", "payment_value"]
].describe()

Descriptive Statistics - Payments


,payment_installments,payment_value
count,"103,886.00","103,886.00"
mean,2.85,154.10
std,2.69,217.49
min,0.00,0.00
25%,1.00,56.79
50%,1.00,100.00
75%,4.00,171.84
max,24.00,"13,664.08"


In [14]:
order_status = orders["order_status"].value_counts()

print("Order Status Distribution:")
print(order_status)

Order Status Distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [15]:
order_status_percentage = (
    orders["order_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Order Status Percentage:")
print(order_status_percentage)

Order Status Percentage:
order_status
delivered     97.02
shipped        1.11
canceled       0.63
unavailable    0.61
invoiced       0.32
processing     0.30
created        0.01
approved       0.00
Name: proportion, dtype: float64


In [16]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_orders = (
    orders.groupby("purchase_month")
    .size()
    .reset_index(name="order_count")
)

monthly_orders["purchase_month"] = (
    monthly_orders["purchase_month"]
    .astype(str)
)

print(monthly_orders)

   purchase_month  order_count
0         2016-09            4
1         2016-10          324
2         2016-12            1
3         2017-01          800
4         2017-02         1780
5         2017-03         2682
6         2017-04         2404
7         2017-05         3700
8         2017-06         3245
9         2017-07         4026
10        2017-08         4331
11        2017-09         4285
12        2017-10         4631
13        2017-11         7544
14        2017-12         5673
15        2018-01         7269
16        2018-02         6728
17        2018-03         7211
18        2018-04         6939
19        2018-05         6873
20        2018-06         6167
21        2018-07         6292
22        2018-08         6512
23        2018-09           16
24        2018-10            4


In [17]:
highest_order_month = monthly_orders.loc[
    monthly_orders["order_count"].idxmax()
]

print("Month with highest number of orders:")
print(highest_order_month)

Month with highest number of orders:
purchase_month    2017-11
order_count          7544
Name: 13, dtype: object


In [18]:
lowest_order_month = monthly_orders.loc[
    monthly_orders["order_count"].idxmin()
]

print("\nMonth with lowest number of orders:")
print(lowest_order_month)


Month with lowest number of orders:
purchase_month    2016-12
order_count             1
Name: 2, dtype: object


In [19]:
products_category = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [20]:
category_sales = order_items.merge(
    products_category[
        ["product_id", "product_category_name_english"]
    ],
    on="product_id",
    how="left"
)


In [21]:
category_sales_summary = (
    category_sales
    .groupby("product_category_name_english")
    .agg(
        total_sales=("price", "sum"),
        total_items=("order_item_id", "count")
    )
    .sort_values("total_sales", ascending=False)
)

print(category_sales_summary.head(10))

                               total_sales  total_items
product_category_name_english                          
health_beauty                 1,258,681.34         9670
watches_gifts                 1,205,005.68         5991
bed_bath_table                1,036,988.68        11115
sports_leisure                  988,048.97         8641
computers_accessories           911,954.32         7827
furniture_decor                 729,762.49         8334
cool_stuff                      635,290.85         3796
housewares                      632,248.66         6964
auto                            592,720.11         4235
garden_tools                    485,256.46         4347


In [22]:
print("Bottom 10 categories by sales:")

print(
    category_sales_summary
    .sort_values("total_sales")
    .head(10)
)

Bottom 10 categories by sales:
                               total_sales  total_items
product_category_name_english                          
security_and_services               283.29            2
fashion_childrens_clothes           569.85            8
cds_dvds_musicals                   730.00           14
home_comfort_2                      760.27           30
flowers                           1,110.04           33
diapers_and_hygiene               1,567.59           39
arts_and_craftmanship             1,814.01           24
la_cuisine                        2,054.99           14
fashion_sport                     2,119.51           30
fashio_female_clothing            2,803.64           48


In [23]:
payment_distribution = payments["payment_type"].value_counts()

print("Payment Type Distribution:")
print(payment_distribution)

Payment Type Distribution:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [24]:
payment_percentage = (
    payments["payment_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Payment Type Percentage:")
print(payment_percentage)

Payment Type Percentage:
payment_type
credit_card   73.92
boleto        19.04
voucher        5.56
debit_card     1.47
not_defined    0.00
Name: proportion, dtype: float64


In [25]:
payment_summary = (
    payments
    .groupby("payment_type")
    .agg(
        total_payment=("payment_value", "sum"),
        average_payment=("payment_value", "mean"),
        transactions=("payment_type", "count")
    )
    .sort_values("total_payment", ascending=False)
)

print(payment_summary)

              total_payment  average_payment  transactions
payment_type                                              
credit_card   12,542,084.19           163.32         76795
boleto         2,869,361.27           145.03         19784
voucher          379,436.87            65.70          5775
debit_card       217,989.79           142.57          1529
not_defined            0.00             0.00             3


In [26]:
review_distribution = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

print("Review Score Distribution:")
print(review_distribution)

Review Score Distribution:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [27]:
review_percentage = (
    reviews["review_score"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("Review Score Percentage:")
print(review_percentage)

Review Score Percentage:
review_score
1   11.51
2    3.18
3    8.24
4   19.29
5   57.78
Name: proportion, dtype: float64


In [28]:
average_review = reviews["review_score"].mean()

print("Average Review Score:", round(average_review, 2))

Average Review Score: 4.09


In [30]:
delivery_data = orders[
    orders["order_status"] == "delivered"
].copy()

In [32]:
delivery_data["delivery_days"] = (
    delivery_data["order_delivered_customer_date"]
    - delivery_data["order_purchase_timestamp"]
).dt.days
print(
    delivery_data["delivery_days"].describe()
)

count   96,470.00
mean        12.09
std          9.55
min          0.00
25%          6.00
50%         10.00
75%         15.00
max        209.00
Name: delivery_days, dtype: float64


In [33]:
average_delivery_days = delivery_data["delivery_days"].mean()

print(
    "Average Delivery Time:",
    round(average_delivery_days, 2),
    "days"
)

Average Delivery Time: 12.09 days


In [35]:
delivery_data["delivery_delay_days"] = (
    delivery_data["order_delivered_customer_date"]
    - delivery_data["order_estimated_delivery_date"]
).dt.days
print(
    delivery_data["delivery_delay_days"].describe()
)

count   96,470.00
mean       -11.88
std         10.18
min       -147.00
25%        -17.00
50%        -12.00
75%         -7.00
max        188.00
Name: delivery_delay_days, dtype: float64


In [36]:
delayed_orders = (
    delivery_data["delivery_delay_days"] > 0
).sum()

total_delivered_orders = len(delivery_data)

delay_percentage = (
    delayed_orders / total_delivered_orders
) * 100

print("Delayed Orders:", delayed_orders)
print("Total Delivered Orders:", total_delivered_orders)
print(
    "Delayed Order Percentage:",
    round(delay_percentage, 2),
    "%"
)

Delayed Orders: 6534
Total Delivered Orders: 96478
Delayed Order Percentage: 6.77 %


In [37]:
correlation_data = order_analysis[
    [
        "total_sales",
        "total_freight",
        "total_items"
    ]
].copy()
correlation_matrix = correlation_data.corr()

print("Correlation Matrix:")
print(correlation_matrix.round(2))

Correlation Matrix:
               total_sales  total_freight  total_items
total_sales           1.00           0.41         0.15
total_freight         0.41           1.00         0.44
total_items           0.15           0.44         1.00


In [38]:
Q1 = order_analysis["total_sales"].quantile(0.25)
Q3 = order_analysis["total_sales"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower Bound:", round(lower_bound, 2))
print("Upper Bound:", round(upper_bound, 2))

Lower Bound: -110.1
Upper Bound: 305.9


In [39]:
sales_outliers = order_analysis[
    (order_analysis["total_sales"] < lower_bound) |
    (order_analysis["total_sales"] > upper_bound)
]

print(
    "Number of Sales Outliers:",
    len(sales_outliers)
)

Number of Sales Outliers: 7913


In [40]:
outlier_percentage = (
    len(sales_outliers) /
    len(order_analysis)
) * 100

print(
    "Outlier Percentage:",
    round(outlier_percentage, 2),
    "%"
)

Outlier Percentage: 7.96 %


In [41]:
print(
    sales_outliers
    .sort_values("total_sales", ascending=False)
    [["order_id", "total_sales", "total_items"]]
    .head(10)
)

                               order_id  total_sales  total_items
13390  03caa2c082116e1d31e67e9ae3700499    13,440.00         8.00
66599  736e1922ae60d0d6a89247b851902527     7,160.00         4.00
22171  0812eb902a67711a1cb742b3cdaa65ae     6,735.00         1.00
28326  fefacc66af859508bf1a7934eab1e97f     6,729.00         1.00
3508   f5136e38d1a14a4dbd87dff67da82701     6,499.00         1.00
32322  2cc9089445046817a7539d90805e6e5a     5,934.60         6.00
53352  a96610ab360d42a2e5335a3998b4718a     4,799.00         1.00
40327  199af31afc78c699f0dbf71fb178d4d4     4,690.00         1.00
41086  b4c4b76c642808cbe472a32b86cddc95     4,599.90         2.00
40342  8dbc85d1447242f3b127dda390d56e19     4,590.00         1.00


In [42]:
print("-----EDA SUMMARY ------")

print(
    "Total Orders:",
    len(orders)
)

print(
    "Total Customers:",
    len(customers)
)

print(
    "Total Products:",
    len(products)
)

print(
    "Total Sellers:",
    len(sellers)
)

print(
    "Average Review Score:",
    round(average_review, 2)
)

print(
    "Average Delivery Time:",
    round(average_delivery_days, 2),
    "days"
)

print(
    "Delayed Order Percentage:",
    round(delay_percentage, 2),
    "%"
)

print(
    "Sales Outlier Percentage:",
    round(outlier_percentage, 2),
    "%"
)

-----EDA SUMMARY ------
Total Orders: 99441
Total Customers: 99441
Total Products: 32951
Total Sellers: 3095
Average Review Score: 4.09
Average Delivery Time: 12.09 days
Delayed Order Percentage: 6.77 %
Sales Outlier Percentage: 7.96 %
